In [18]:
import json
import logging
from typing import Dict, Any
from google.cloud import storage

# ================= 配置区域 (已根据你的截图填好) =================
BUCKET_NAME = "xhs-humor-data"          # 你的 Bucket 名称
SOURCE_BLOB_NAME = "chime_full.json"    # 你的文件名
OUTPUT_FILENAME = "chime_rag_ready.jsonl" # 输出结果文件名

# 设置日志
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def format_chime_item(item: Dict[str, Any]) -> Dict[str, Any]:
    """
    核心转换逻辑：将原始 JSON 对象转换为 RAG 文本块 + 元数据
    """
    # 1. 字段提取与默认值处理
    meme = item.get('meme', '未知梗')
    meaning = item.get('meaning', '暂无含义解释')
    origin = item.get('origin', '未知来源')
    
    # 2. 处理例句 (兼顾字符串和列表格式)
    examples = item.get('example', [])
    if isinstance(examples, list):
        examples_str = "\n".join([f"- {ex}" for ex in examples])
    elif isinstance(examples, str):
        examples_str = f"- {examples}"
    else:
        examples_str = "暂无例句"

    # 3. 构建语义文本 (Page Content) - 这是 RAG 检索的关键
    # 格式化为类似百科词条的文本，有助于 Embedding 模型理解语义
    text_content = (
        f"网络热梗：{meme}\n"
        f"含义解析：{meaning}\n"
        f"起源/出处：{origin}\n"
        f"使用例句：\n{examples_str}"
    )

    # 4. 构建元数据 (Metadata) - 用于检索时的 Filter
    metadata = {
        "source": "chime_dataset",
        "meme_type": item.get('type_cn', '其他'), # 保留中文分类
        "is_offensive": item.get('offense', False) # 标记是否冒犯，便于过滤
    }

    return {
        "page_content": text_content,
        "metadata": metadata
    }

def load_from_gcs(bucket_name, blob_name):
    """直接从 Google Cloud Storage 读取文件"""
    try:
        logging.info(f"正在连接 GCS Bucket: {bucket_name} ...")
        storage_client = storage.Client()
        bucket = storage_client.bucket(bucket_name)
        blob = bucket.blob(blob_name)
        
        logging.info(f"正在下载文件: {blob_name} ...")
        content = blob.download_as_text(encoding='utf-8')
        return json.loads(content)
    except Exception as e:
        logging.error(f"从 GCS 下载或解析失败: {e}")
        raise e

def main():
    try:
        # 1. 从 GCS 读取数据
        raw_data = load_from_gcs(BUCKET_NAME, SOURCE_BLOB_NAME)
        logging.info(f"下载成功，共获取 {len(raw_data)} 条原始数据")

        # 2. 处理数据
        logging.info("开始进行数据清洗与格式化...")
        processed_docs = []
        for item in raw_data:
            # 简单过滤：如果没有含义解释，可能对 RAG 价值不大，可以选择跳过
            if not item.get('meaning'):
                continue
            processed_docs.append(format_chime_item(item))

        # 3. 写入本地 JSONL 文件
        logging.info(f"处理完成，正在写入输出文件: {OUTPUT_FILENAME}")
        with open(OUTPUT_FILENAME, 'w', encoding='utf-8') as f:
            for doc in processed_docs:
                f.write(json.dumps(doc, ensure_ascii=False) + '\n')
        
        logging.info("✅ 转换成功！文件已准备好用于 RAG 向量化。")
        
        # 打印一条预览
        print("\n--- 转换结果预览 (第一条) ---")
        print(json.dumps(processed_docs[0], ensure_ascii=False, indent=2))

    except Exception as e:
        logging.error(f"程序执行出错: {e}")

if __name__ == "__main__":
    main()

2026-02-17 01:52:44,214 - INFO - 正在连接 GCS Bucket: xhs-humor-data ...
2026-02-17 01:52:44,227 - INFO - 正在下载文件: chime_full.json ...
2026-02-17 01:52:44,482 - INFO - 下载成功，共获取 1458 条原始数据
2026-02-17 01:52:44,483 - INFO - 开始进行数据清洗与格式化...
2026-02-17 01:52:44,504 - INFO - 处理完成，正在写入输出文件: chime_rag_ready.jsonl
2026-02-17 01:52:44,622 - INFO - ✅ 转换成功！文件已准备好用于 RAG 向量化。



--- 转换结果预览 (第一条) ---
{
  "page_content": "网络热梗：treetree的\n含义解析：\"treetree的\"是一个谐音梗，通常用来形容食物或物品的口感或外观上的“脆脆”感觉。\n起源/出处：源于吃播，在直播中主播因为口音或习惯将“脆脆”发音为“tree tree”，之后被网友在评论区中玩梗并传播开来，尤其在抖音等平台上常见。\n使用例句：\n",
  "metadata": {
    "source": "chime_dataset",
    "meme_type": "谐音",
    "is_offensive": false
  }
}
